In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41590-021-00922-4"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41590_2021_922_MOESM2_ESM.xlsx"
table_name = "clean_degs.xlsx" # local name # using this for paper_degs

## Read in file

In [57]:
excel = pd.read_excel(table_name, sheet_name= ["Cluster_Markers_CD45PosNeg","Cluster_Markers_Cellype_NK_ILCs", "Cluster_Markers_Celltype_Myeloi", "DEG_Celltype_LeanvsObese"] ,skiprows = 2)

In [58]:
for e in list(excel.values()):
    print(e.columns)

Index(['p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj', 'cluster', 'gene'], dtype='object')
Index(['p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj', 'cluster', 'gene'], dtype='object')
Index(['p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj', 'cluster', 'gene'], dtype='object')
Index(['gene', 'p_val', 'avg_logFC', 'pct.1 (Lean)', 'pct.2 (Obese)',
       'p_val_adj', 'celltype'],
      dtype='object')


In [59]:
cols = [
    {
        "sheet_name": "Cluster_Markers_CD45PosNeg",
        "cols": {
            "p_val": "p_raw",
            "p_val_adj": "p_corr",
            "avg_logFC": "logfc",
            "cluster": "cell_type_label",
            "gene": "feature_name"
        }
    },
    {
        "sheet_name": "Cluster_Markers_Cellype_NK_ILCs",
        "cols": {
            "p_val": "p_raw",
            "p_val_adj": "p_corr",
            "avg_logFC": "logfc",
            "cluster": "cell_type_label",
            "gene": "feature_name"
        }
    },
    {
        "sheet_name": "Cluster_Markers_Celltype_Myeloi",
        "cols": {
            "p_val": "p_raw",
            "p_val_adj": "p_corr",
            "avg_logFC": "logfc",
            "cluster": "cell_type_label",
            "gene": "feature_name"
        }
    },
    {
        "sheet_name": "DEG_Celltype_LeanvsObese",
        "cols": {
            "p_val": "p_raw",
            "p_val_adj": "p_corr",
            "avg_logFC": "logfc",
            "celltype": "cell_type_label",
            "gene": "feature_name"
        }
    },
]

In [60]:
for sheet in cols:
    sname = sheet["sheet_name"]
    print(sname)
    # only keep the columns we want
    excel[sname] = excel[sname].loc[:,sheet["cols"].keys()].rename(columns=sheet["cols"])
    # add sheet name to the dataframe
    excel[sname]["sheet_name"] = sname

Cluster_Markers_CD45PosNeg
Cluster_Markers_Cellype_NK_ILCs
Cluster_Markers_Celltype_Myeloi
DEG_Celltype_LeanvsObese


In [61]:
df = pd.concat(list(excel.values()))

In [62]:
df.tail()

,p_raw,p_corr,logfc,cell_type_label,feature_name,sheet_name
6035,2.649478e-105,7.509679e-101,0.747201,trNK,IFITM1,DEG_Celltype_LeanvsObese
6036,5.087963e-37,1.442132e-32,0.861339,trNK,CSF2,DEG_Celltype_LeanvsObese
6037,5.664293e-115,1.605487e-110,0.879374,trNK,RPS4Y1,DEG_Celltype_LeanvsObese
6038,2.595948e-112,7.357956e-108,0.879925,trNK,DDX3Y,DEG_Celltype_LeanvsObese
6039,3.266413e-119,9.258320e-115,2.183736,trNK,MTRNR2L8,DEG_Celltype_LeanvsObese


## Unfiltered

In [65]:
unfiltered_df.head()

,p_raw,p_corr,logfc,cell_type_label,feature_name,sheet_name,organism,cell_source,cell_state,feature_identifier
0,0.000000e+00,0.000000e+00,0.911908,CD8 gdT,LINC02446,Cluster_Markers_CD45PosNeg,homo_sapiens,adipose,None,None
1,0.000000e+00,0.000000e+00,0.905986,CD8 gdT,LEF1,Cluster_Markers_CD45PosNeg,homo_sapiens,adipose,None,None
2,0.000000e+00,0.000000e+00,0.748453,CD8 gdT,FXYD2,Cluster_Markers_CD45PosNeg,homo_sapiens,adipose,None,None
3,0.000000e+00,0.000000e+00,0.727598,CD8 gdT,ZNF683,Cluster_Markers_CD45PosNeg,homo_sapiens,adipose,None,None
4,4.540614e-241,1.242811e-236,0.962575,CD8 gdT,KLRC2,Cluster_Markers_CD45PosNeg,homo_sapiens,adipose,None,None


In [66]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None
unfiltered_df.rename(columns={"gene": "feature_name"}, inplace=True)
unfiltered_df.rename(columns={"cluster": "cell_type_label"}, inplace=True)

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({
        "extracted": {
            "organism": row["organism"], 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene"
            },
        "derived": {
            "organism": row["organism"], 
            "cell_type_id": None, 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene", 
            "feature_identifier": row["feature_identifier"], 
            "feature_type": "ensembl"
            },
        "source": {
            "source_type": "deg", 
            "source_rationale": "unfiltered", 
            "source_id": row["sheet_name"],
            "source_metrics": {
                "p_corr": row["p_corr"],
                "log_fc": row["logfc"],
            }
            }
        })

result[:1]

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'CD8 gdT',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'LINC02446',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'CD8 gdT',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'LINC02446',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'Cluster_Markers_CD45PosNeg',
   'source_metrics': {'p_corr': 0.0, 'log_fc': 0.911907662268655}}}]

## Save Unfiltered

In [67]:
with open("evidence_unfiltered_metrics.json", "w") as f:
    json.dump(result, f, indent = 4) 

## Perform the filtering

In [15]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [16]:

min_mean = 50
max_pval = 1e-10
min_lfc = 2
max_gene_shares = 10
max_per_celltype = 50

"""
min_mean = 0
max_pval = 1
min_lfc = 0
max_gene_shares = 100
max_per_celltype = 100
"""

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [17]:
markers

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
25593,0.000000e+00,2.495286,0.166,0.003,0.000000e+00,B cell,IGHG3
25591,0.000000e+00,3.038060,0.173,0.003,0.000000e+00,B cell,IGHG1
25592,0.000000e+00,2.512809,0.193,0.001,0.000000e+00,B cell,IGLC3
23889,0.000000e+00,2.152270,0.283,0.003,0.000000e+00,Smooth muscle,THBS4
25588,0.000000e+00,3.649848,0.289,0.012,0.000000e+00,B cell,IGHA1
...,...,...,...,...,...,...,...
20174,0.000000e+00,3.062876,1.000,0.126,0.000000e+00,Preadipocyte,MGP
14585,0.000000e+00,2.210930,1.000,0.721,0.000000e+00,ncMo,SAT1
14589,0.000000e+00,2.106703,1.000,0.387,0.000000e+00,ncMo,COTL1
10648,1.009903e-241,2.551416,1.000,0.338,2.764206e-237,cDC1,HLA-DRB1


In [18]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
B cell           0.437200
Endothelial      0.796929
Mo-2             0.803667
Smooth muscle    0.823375
Myeloid-like     0.829450
Mo-1             0.857000
NK-like          0.859000
Neu              0.886000
LAM              0.886833
IM               0.899571
PVM              0.899618
cDC2A            0.917750
cDC2B            0.933059
ncMo             0.934471
APC              0.941500
mNK              0.950700
Preadipocyte     0.951235
cDC1             0.978538
ILCs             0.990000
Name: pct.1, dtype: float64

In [19]:
markers[col_ct].value_counts().sort_index()

cluster
APC               8
B cell           10
Endothelial      28
ILCs              1
IM                7
LAM               6
Mo-1              1
Mo-2              3
Myeloid-like     20
NK-like           2
Neu               6
PVM              34
Preadipocyte     17
Smooth muscle    16
cDC1             13
cDC2A             4
cDC2B            17
mNK              10
ncMo             17
Name: count, dtype: int64

## Convert to evidence json


In [21]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = None

result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'B cell',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG3',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'B cell',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG3',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'filtered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'B cell',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG1',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'B cell',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG1',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale

## Save evidence

In [22]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 